# 5부 자습 노트북 — 그래디언트 부스팅 (GBM)

본 노트북은 *직접 실행하며* 학습하는 자료이다. 5부 이론 교재(`part5_GBM_HARD_이론.md`)의 핵심 코드를 모두 돌려볼 수 있다.

**시리즈에서 본 부의 위치**: 4부 AdaBoost의 *지수 손실 한계*를 *손실함수 일반화*로 풀어내는 본 시리즈의 *클라이맥스*. 캘리포니아 R² 0.23 → 0.65의 *회복 사례*가 본 부의 핵심 사건이다.

**데이터셋**:
- **Ames** (2,930 × 82) — 정제된 데이터, GBM이 RF를 미세하게 능가
- **California** (20,640 × 9) — capped 이상치 데이터, Huber 손실로 회복

## 환경 준비와 데이터 로딩

In [ ]:
# Colab 등에서 처음 한 번만 실행
# !pip install koreanize-matplotlib --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["axes.unicode_minus"] = False

URL_AMES  = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/AmesHousing.csv"
URL_CALIF = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"

try:
    ames_raw  = pd.read_csv(URL_AMES)
    calif_raw = pd.read_csv(URL_CALIF)
except Exception:
    rng = np.random.default_rng(42)
    n = 2900
    ames_raw = pd.DataFrame({
        "Overall Qual": rng.integers(1, 11, n),
        "Gr Liv Area":  rng.integers(500, 4500, n),
        "Year Built":   rng.integers(1900, 2010, n),
    })
    ames_raw["SalePrice"] = (
        50000 + ames_raw["Overall Qual"] * 25000
        + ames_raw["Gr Liv Area"] * 60 + rng.normal(0, 20000, n)
    ).astype(int)
    
    n2 = 5000
    calif_raw = pd.DataFrame({
        "longitude":      rng.uniform(-124, -114, n2),
        "latitude":       rng.uniform(32, 42, n2),
        "housing_median_age": rng.integers(1, 52, n2),
        "total_rooms":    rng.integers(2, 6000, n2),
        "population":     rng.integers(3, 5000, n2),
        "households":     rng.integers(2, 1000, n2),
        "median_income":  rng.uniform(0.5, 15, n2),
        "median_house_value": np.clip(rng.normal(200000, 100000, n2), 15000, 500001).astype(int)
    })

print(f"Ames:       {ames_raw.shape}")
print(f"California: {calif_raw.shape}")

In [ ]:
def prepare_ames(df_in):
    df = df_in.copy()
    df = df.drop(columns=[c for c in ["Order", "PID"] if c in df.columns])
    for c in ["Pool QC", "Misc Feature", "Alley", "Fence", "Fireplace Qu",
              "Garage Qual", "Garage Cond", "Bsmt Qual", "Bsmt Cond"]:
        if c in df.columns:
            df[c] = df[c].fillna("None")
    num = df.select_dtypes("number").columns
    df[num] = df[num].fillna(df[num].median())
    df = df[df["Gr Liv Area"] < 4000].copy()
    if all(c in df.columns for c in ["1st Flr SF", "2nd Flr SF", "Total Bsmt SF"]):
        df["Total SF"] = df["1st Flr SF"] + df["2nd Flr SF"] + df["Total Bsmt SF"]
    return df

ames = prepare_ames(ames_raw)
y_ames = np.log1p(ames["SalePrice"])
X_ames = ames.select_dtypes("number").drop(columns=["SalePrice"])

calif = calif_raw.copy()
if "ocean_proximity" in calif.columns:
    calif = pd.get_dummies(calif, columns=["ocean_proximity"], drop_first=True)
num = calif.select_dtypes("number").columns
calif[num] = calif[num].fillna(calif[num].median())
y_calif = calif["median_house_value"]
X_calif = calif.drop(columns=["median_house_value"])

print(f"X_ames:  {X_ames.shape}")
print(f"X_calif: {X_calif.shape}")

---
## 0장 GBM의 동기 — 왜 손실함수 일반화인가

GBM의 핵심 아이디어 한 줄:

> *부스팅을 손실함수의 그래디언트 위에서 정의하면, 어떤 손실함수도 사용할 수 있다.*

지수 손실에 갇힌 AdaBoost가 *제곱·절대·Huber·로그 손실* 등 *모든 미분 가능 손실*로 일반화된다.

---
## 1장 잔차 학습 — 제곱 손실 GBM의 직관

GBM(제곱 손실)은 *잔차를 학습 타깃으로 받는 새 트리*를 누적한다.

알고리즘:
1. 초기 모델 F_0 = 모든 y의 평균
2. 잔차 r = y - F_t 계산
3. 새 트리 h_t가 r을 학습
4. F_{t+1} = F_t + η · h_t
5. 2~4 반복

In [ ]:
# 잔차 학습 직접 시뮬레이션 — sklearn 도움 없이 GBM의 핵심을 손으로 구현
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X_ames, y_ames, test_size=0.3, random_state=42)
y_tr = y_tr.values
y_te = y_te.values

# 초기 모델 — 평균
F0 = y_tr.mean()
F_tr = np.full(len(y_tr), F0)
F_te = np.full(len(y_te), F0)
lr = 0.1
n_rounds = 50

trees = []
for t in range(n_rounds):
    # 잔차 계산
    residual = y_tr - F_tr
    
    # 잔차를 학습 타깃으로 새 트리
    h_t = DecisionTreeRegressor(max_depth=3, random_state=t)
    h_t.fit(X_tr, residual)
    trees.append(h_t)
    
    # 모델 업데이트
    F_tr = F_tr + lr * h_t.predict(X_tr)
    F_te = F_te + lr * h_t.predict(X_te)

print(f"수동 GBM 50라운드 결과:")
print(f"  학습 R²: {r2_score(y_tr, F_tr):.4f}")
print(f"  검증 R²: {r2_score(y_te, F_te):.4f}")
print(f"\n→ 잔차 학습이 부스팅의 본질이다.")

---
## 2장 누적의 위력 — 평균에서 정교한 함수까지

학습률 η와 라운드 수 T의 트레이드오프. 작은 학습률 + 많은 라운드가 좋다.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

print(f"{'lr':>6s}  {'n_est':>8s}  {'CV R²':>10s}")
print("-" * 28)
for lr, n_est in [(0.5, 50), (0.5, 200), (0.1, 100), (0.1, 500), (0.05, 200), (0.05, 1000)]:
    gbm = GradientBoostingRegressor(
        n_estimators=n_est, learning_rate=lr, max_depth=3, random_state=42
    )
    r2 = cross_val_score(gbm, X_ames, y_ames, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"{lr:>6.2f}  {n_est:>8d}  {r2:>10.4f}")

print("\n→ 작은 학습률 + 많은 라운드가 더 정밀하다.")

---
## 3장 일반 그래디언트 부스팅 — 손실함수 일반화

`loss` 매개변수만 바꾸면 다양한 손실함수를 쓸 수 있다.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

print("=== 손실함수별 R² (Ames) ===")
print(f"{'loss':<20s}  {'CV R²':>10s}")
print("-" * 32)
for loss in ["squared_error", "absolute_error", "huber"]:
    gbm = GradientBoostingRegressor(
        loss=loss, n_estimators=200, learning_rate=0.1,
        max_depth=3, random_state=42
    )
    r2 = cross_val_score(gbm, X_ames, y_ames, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"{loss:<20s}  {r2:>10.4f}")

---
## 4장 분위수 회귀 — 예측 구간 만들기

`loss="quantile"`로 *특정 분위수*를 예측. 0.1, 0.5, 0.9 세 모델로 *80% 예측 구간* 생성.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

X_tr, X_te, y_tr, y_te = train_test_split(X_ames, y_ames, test_size=0.3, random_state=42)

# 0.1, 0.5, 0.9 세 분위수 모델
gbms = {}
for alpha in [0.1, 0.5, 0.9]:
    gbm = GradientBoostingRegressor(
        loss="quantile", alpha=alpha,
        n_estimators=200, learning_rate=0.1,
        random_state=42
    )
    gbm.fit(X_tr, y_tr)
    gbms[alpha] = gbm

# 첫 10개 테스트 샘플의 예측 구간
print(f"{'i':>3s}  {'실제':>8s}  {'0.1 분위':>10s}  {'중앙값':>10s}  {'0.9 분위':>10s}  {'in?':>4s}")
print("-" * 56)
y_te_arr = y_te.values
n_in = 0
for i in range(20):
    low  = gbms[0.1].predict(X_te.iloc[i:i+1])[0]
    med  = gbms[0.5].predict(X_te.iloc[i:i+1])[0]
    high = gbms[0.9].predict(X_te.iloc[i:i+1])[0]
    actual = y_te_arr[i]
    in_interval = low <= actual <= high
    if in_interval:
        n_in += 1
    mark = "✓" if in_interval else "✗"
    print(f"{i:>3d}  {actual:>8.3f}  {low:>10.3f}  {med:>10.3f}  {high:>10.3f}  {mark:>4s}")

print(f"\n20개 중 {n_in}개가 80% 예측 구간 안에 있음")

---
## 5장 GBM의 핵심 매개변수

5종 매개변수 — `n_estimators`, `learning_rate`, `max_depth`, `subsample`, `loss` — 의 효과.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# subsample (확률적 GBM) 효과
print(f"{'subsample':>10s}  {'CV R²':>10s}")
print("-" * 24)
for sub in [0.5, 0.7, 0.8, 1.0]:
    gbm = GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.1,
        max_depth=3, subsample=sub, random_state=42
    )
    r2 = cross_val_score(gbm, X_ames, y_ames, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"{sub:>10.2f}  {r2:>10.4f}")

---
## 6장 Ames에서 GBM — RF를 미세하게 능가

GBM(튜닝) > RF > GBM(기본) > 단일 트리 순.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

print(f"{'모델':<30s}  {'R²':>10s}")
print("-" * 42)
models = [
    ("단일 트리 (튜닝)",       DecisionTreeRegressor(max_depth=7, min_samples_leaf=10, random_state=42)),
    ("랜덤 포레스트",          RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    ("GBM (기본)",             GradientBoostingRegressor(random_state=42)),
    ("GBM (튜닝)",             GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42)),
]
for name, m in models:
    r2 = cross_val_score(m, X_ames, y_ames, cv=5, scoring="r2", n_jobs=-1).mean()
    print(f"{name:<30s}  {r2:>10.4f}")

### staged_predict — 라운드별 R² 추적

In [ ]:
from sklearn.metrics import r2_score

X_tr, X_te, y_tr, y_te = train_test_split(X_ames, y_ames, test_size=0.3, random_state=42)

gbm = GradientBoostingRegressor(n_estimators=500, learning_rate=0.1,
                                  max_depth=3, random_state=42)
gbm.fit(X_tr, y_tr)
test_r2 = np.array([r2_score(y_te, p) for p in gbm.staged_predict(X_te)])

best_round = np.argmax(test_r2) + 1
print(f"최적 라운드: {best_round}, 최대 R²: {test_r2.max():.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(test_r2, color="#1F3A5F", linewidth=2)
ax.axvline(best_round - 1, color="#C0392B", linestyle="--", label=f"최적 = {best_round}")
ax.set_xlabel("라운드")
ax.set_ylabel("Test R²")
ax.set_title("GBM 라운드별 Test R² — 약 200~500 라운드에서 정점")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7장 캘리포니아 회복 — AdaBoost(0.23) → GBM(0.65)

본 시리즈의 *클라이맥스*. 같은 데이터, 같은 알고리즘 골격, 다른 손실함수만으로 R²가 *2.8배 점프*한다.

In [ ]:
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor

print("=== 캘리포니아 — 손실함수의 위력 ===")
print(f"{'모델':<30s}  {'R²':>10s}")
print("-" * 42)

# AdaBoost (지수 손실 고정)
ab = AdaBoostRegressor(n_estimators=100, random_state=42)
r2 = cross_val_score(ab, X_calif, y_calif, cv=3, scoring="r2", n_jobs=-1).mean()
print(f"{'AdaBoost (지수 손실)':<30s}  {r2:>10.4f}")

# GBM 다양한 손실
for loss in ["squared_error", "absolute_error", "huber"]:
    gbm = GradientBoostingRegressor(
        loss=loss, n_estimators=200, learning_rate=0.1, random_state=42
    )
    r2 = cross_val_score(gbm, X_calif, y_calif, cv=3, scoring="r2", n_jobs=-1).mean()
    label = f"GBM ({loss})"
    print(f"{label:<30s}  {r2:>10.4f}")

print("\n→ AdaBoost 0.23 → GBM Huber 0.65. 손실함수 선택만으로!")

### 왜 Huber가 최고인가 — 캘리포니아의 capped 이상치

In [ ]:
# capped 봉우리 시각화
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_calif, bins=80, color="#1F3A5F", alpha=0.7)
ax.axvline(500001, color="#C0392B", linestyle="--", linewidth=2, label="$500,001 cap")
ax.set_xlabel("median_house_value")
ax.set_ylabel("빈도")
ax.set_title("캘리포니아 가격 분포 — 오른쪽 끝 봉우리가 capped 이상치")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"capped 비율: {(y_calif == 500001).sum()}건 ({(y_calif == 500001).mean()*100:.2f}%)")
print()
print("Huber 손실: 작은 오차는 제곱처럼, 큰 오차는 절대값처럼.")
print("           capped 이상치의 큰 오차에 *과도하게 끌리지 않음*.")

---
## 8장 XGBoost로의 다리 — 왜 더 빠른 구현이 필요한가

GBM은 *정확도는 좋지만 학습이 느리다* (순차 학습). XGBoost는 같은 알고리즘 골격에 *대규모 최적화*를 더했다.

In [ ]:
import time
from sklearn.ensemble import GradientBoostingRegressor

# sklearn GBM 학습 시간
t0 = time.time()
gbm = GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42)
gbm.fit(X_ames, y_ames)
t_sklearn = time.time() - t0
print(f"sklearn GBM: {t_sklearn:.2f}초")

# XGBoost (설치되어 있다면)
try:
    import xgboost as xgb
    t0 = time.time()
    xgb_model = xgb.XGBRegressor(
        n_estimators=200, max_depth=3,
        tree_method="hist",     # 히스토그램 분할 — 빠름
        n_jobs=-1,              # 모든 CPU
        random_state=42
    )
    xgb_model.fit(X_ames, y_ames)
    t_xgb = time.time() - t0
    print(f"XGBoost:     {t_xgb:.2f}초")
    print(f"속도 비율:   {t_sklearn / t_xgb:.1f}배 빠름")
except ImportError:
    print("XGBoost 미설치. !pip install xgboost로 설치 후 다시 실행.")
    print("6부에서 XGBoost·LightGBM·CatBoost를 자세히 다룬다.")

---
## 마무리

본 노트북에서 *직접 실행*한 GBM의 핵심을 한 표로 정리한다.

| 장 | 핵심 도구 |
|---|---|
| 0장 GBM 동기 | 손실함수 일반화의 의미 |
| 1장 잔차 학습 | 수동 GBM 시뮬레이션 |
| 2장 누적의 위력 | learning_rate × n_estimators 트레이드오프 |
| 3장 일반 부스팅 | `loss="squared_error", "huber"` 등 |
| 4장 분위수 회귀 | `loss="quantile"`, alpha=0.1/0.5/0.9 |
| 5장 매개변수 | `subsample=0.8` (확률적 GBM) |
| 6장 Ames | GBM(튜닝) R² 0.89 — RF 미세 능가 |
| 7장 캘리포니아 회복 | AdaBoost 0.23 → GBM Huber 0.65 |
| 8장 XGBoost 다리 | `tree_method="hist"`, `n_jobs=-1` |

### 다음 단계

6부 자습 노트북(*예정*)에서 **XGBoost · LightGBM · CatBoost**를 다룬다. 본 부에서 익힌 *손실함수 선택*과 *튜닝 직관*이 현대 캐글 부스팅 삼대장에 그대로 적용된다.